# 02 · 실제 DeepAgents 연결 (선택 심화)
이 단계는 기본 실습의 완료 조건이 아닙니다. 실제 Harness는 `deepagents`, `langchain-openai`, LiteLLM OpenAI-compatible endpoint, MCP Registry 및 각 서비스의 인증이 모두 필요합니다. AI Core native SDK의 `chat.completions.create`만으로는 production DeepAgents runtime을 그대로 대체할 수 없습니다.

<div style="margin: 18px 0 12px; padding: 13px 15px; border-left: 4px solid #2e7d32; background: #e8f5e9; border-radius: 6px; color: #202124; box-shadow: 0 1px 2px rgba(60,64,67,.12);">
  <div style="font-size: .82em; font-weight: 700; color: #2e7d32; letter-spacing: .02em;">🖥️  UI 실행</div>
  <div style="margin-top: 8px; font-size: .94em; line-height: 1.6;">
<h2 style="margin: 0 0 8px; font-size: 1.05em; line-height: 1.35; color: #202124;">UI: DeepAgents 연결 준비 상태</h2>
<p style="margin: 7px 0;">이 UI는 prompt를 읽고 외부 runtime 연결에 필요한 항목의 준비 여부만 점검합니다. 실제 DeepAgent 실행이나 MCP 호출은 하지 않습니다.</p>
  </div>
</div>


In [ ]:
from pathlib import Path
import importlib.util
import os
import sys
import gradio as gr

project_root = next(path for path in [Path.cwd(), *Path.cwd().parents] if (path / 'shared').exists())
sys.path[:0] = [str(project_root / 'shared'), str(project_root / '5_harness')]
from src.harness_prompts import default_prompt, prompt_catalog

prompt_ids = [item['id'] for item in prompt_catalog()]

def inspect_deepagents_setup(prompt_id):
    readiness = {
        'deepagents_installed': importlib.util.find_spec('deepagents') is not None,
        'langchain_openai_installed': importlib.util.find_spec('langchain_openai') is not None,
        'HARNESS_LLM_BASE_URL_set': bool(os.getenv('HARNESS_LLM_BASE_URL')),
        'HARNESS_LLM_API_KEY_set': bool(os.getenv('HARNESS_LLM_API_KEY')),
        'HARNESS_MODEL_set': bool(os.getenv('HARNESS_MODEL')),
        'note': '이 화면은 외부 runtime이나 MCP를 호출하지 않습니다.',
    }
    return default_prompt(prompt_id), readiness

with gr.Blocks() as deepagents_ui:
    gr.Markdown('## DeepAgents Readiness Playground')
    prompt_id = gr.Dropdown(choices=prompt_ids, value='deep_orchestrator', label='Harness prompt')
    check = gr.Button('Prompt와 준비 상태 확인', variant='primary')
    prompt = gr.Textbox(label='Prompt content', lines=18)
    readiness = gr.JSON(label='Readiness')
    check.click(inspect_deepagents_setup, inputs=prompt_id, outputs=[prompt, readiness])
deepagents_ui.launch(inline=True, share=False, show_error=True, server_name='127.0.0.1', height=900)


<div style="margin: 18px 0 12px; padding: 13px 15px; border-left: 4px solid #1a73e8; background: #e8f0fe; border-radius: 6px; color: #202124; box-shadow: 0 1px 2px rgba(60,64,67,.12);">
  <div style="font-size: .82em; font-weight: 700; color: #1a73e8; letter-spacing: .02em;">✅  체크포인트</div>
  <div style="margin-top: 8px; font-size: .94em; line-height: 1.6;">
<h2 style="margin: 0 0 8px; font-size: 1.05em; line-height: 1.35; color: #202124;">체크포인트</h2>
<ul style="margin: 7px 0 0; padding-left: 1.3em;">
<li>원하는 Harness prompt를 UI에서 읽을 수 있다.</li>
<li>선택 의존성과 외부 runtime 설정의 존재 여부를 확인한다.</li>
<li>실제 endpoint와 MCP Registry 권한이 준비되기 전에는 실행하지 않는다.</li>
</ul>
  </div>
</div>


<hr style="border: 0; border-top: 2px solid #c7d2e1; margin: 32px 0 14px;">
<div style="margin: 0 0 18px; color: #4b5f78; font-size: .86em; font-weight: 700; letter-spacing: .04em;">🧩 아래부터 코드 베이스 실습</div>


<div style="margin: 18px 0 12px; padding: 13px 15px; border-left: 4px solid #5f6368; background: #f1f3f4; border-radius: 6px; color: #202124; box-shadow: 0 1px 2px rgba(60,64,67,.12);">
  <div style="font-size: .82em; font-weight: 700; color: #5f6368; letter-spacing: .02em;">🔎  선택 심화</div>
  <div style="margin-top: 8px; font-size: .94em; line-height: 1.6;">
<h2 style="margin: 0 0 8px; font-size: 1.05em; line-height: 1.35; color: #202124;">선택: 코드로 준비 상태와 prompt 살펴보기</h2>
<p style="margin: 7px 0;">아래 셀은 UI가 읽는 prompt catalog, HANA snapshot, 외부 runtime 요구사항을 코드로 확인하는 자료입니다.</p>
  </div>
</div>


<div style="margin: 18px 0 12px; padding: 13px 15px; border-left: 4px solid #1976d2; background: #eef6ff; border-radius: 6px; color: #202124; box-shadow: 0 1px 2px rgba(60,64,67,.12);">
  <div style="font-size: .82em; font-weight: 700; color: #1976d2; letter-spacing: .02em;">📘  개념 · 실습 단계</div>
  <div style="margin-top: 8px; font-size: .94em; line-height: 1.6;">
<h2 style="margin: 0 0 8px; font-size: 1.05em; line-height: 1.35; color: #202124;">1. Harness prompt catalog</h2>
<p style="margin: 7px 0;">prompts 폴더에는 source default prompt 7개가 있습니다. 각 prompt는 orchestrator 또는 specialist의 역할, 권한, 응답 규칙을 정의합니다.</p>
  </div>
</div>


In [ ]:
# Prompt와 delegation 계약을 외부 모듈 없이 작은 dict로 확인합니다.
PROMPTS = {"deep_orchestrator": "요청을 분류하고 specialist에게 읽기 전용 근거 수집을 위임한 뒤 답을 종합한다.", "researcher": "근거와 출처를 수집한다.", "business_agent": "업무 데이터를 읽기 전용으로 분석한다.", "analyst_writer": "근거를 사용자용 답변으로 요약한다."}

def make_agent_plan(request):
    return {"system_prompt": PROMPTS["deep_orchestrator"], "request": request, "delegation": ["researcher", "business_agent"], "final_writer": "analyst_writer", "write_tools": "human_approval_required"}


<div style="margin: 18px 0 12px; padding: 13px 15px; border-left: 4px solid #1976d2; background: #eef6ff; border-radius: 6px; color: #202124; box-shadow: 0 1px 2px rgba(60,64,67,.12);">
  <div style="font-size: .82em; font-weight: 700; color: #1976d2; letter-spacing: .02em;">📘  개념 · 실습 단계</div>
  <div style="margin-top: 8px; font-size: .94em; line-height: 1.6;">
<h2 style="margin: 0 0 8px; font-size: 1.05em; line-height: 1.35; color: #202124;">2. HANA override snapshot</h2>
<p style="margin: 7px 0;">운영 Harness는 HANA SAP_HARNESS_PROMPTS의 override가 있으면 파일 기본값보다 우선합니다. 여기에는 읽기 전용 snapshot만 있으며 HANA에 쓰지 않습니다.</p>
  </div>
</div>


In [ ]:
hana_snapshot = {"source": "SAP_HARNESS_PROMPTS export", "prompts": [{"id": "deep_orchestrator", "content": PROMPTS["deep_orchestrator"]}]}
print(hana_snapshot)


<div style="margin: 18px 0 12px; padding: 13px 15px; border-left: 4px solid #1976d2; background: #eef6ff; border-radius: 6px; color: #202124; box-shadow: 0 1px 2px rgba(60,64,67,.12);">
  <div style="font-size: .82em; font-weight: 700; color: #1976d2; letter-spacing: .02em;">📘  개념 · 실습 단계</div>
  <div style="margin-top: 8px; font-size: .94em; line-height: 1.6;">
<h2 style="margin: 0 0 8px; font-size: 1.05em; line-height: 1.35; color: #202124;">3. Orchestrator prompt</h2>
<p style="margin: 7px 0;">DeepAgents adapter에서는 이 prompt를 system prompt 후보로 사용합니다. prompt만으로 runtime이 완성되는 것은 아니며 도구, 권한, 관찰성도 필요합니다.</p>
  </div>
</div>


In [ ]:
print(PROMPTS["deep_orchestrator"])
print(make_agent_plan("재고 근거를 수집해 발주 초안을 작성해줘."))


<div style="margin: 18px 0 12px; padding: 13px 15px; border-left: 4px solid #1976d2; background: #eef6ff; border-radius: 6px; color: #202124; box-shadow: 0 1px 2px rgba(60,64,67,.12);">
  <div style="font-size: .82em; font-weight: 700; color: #1976d2; letter-spacing: .02em;">📘  개념 · 실습 단계</div>
  <div style="margin-top: 8px; font-size: .94em; line-height: 1.6;">
<h2 style="margin: 0 0 8px; font-size: 1.05em; line-height: 1.35; color: #202124;">4. 선택 의존성</h2>
<p style="margin: 7px 0;">deepagents와 langchain-openai는 기본 실습에 필요하지 않습니다. 설치 여부만 확인합니다.</p>
  </div>
</div>


In [ ]:
runtime_contract = {"optional_packages": ["deepagents", "langgraph", "langchain-openai"], "tools": ["read_only_mcp"], "write_tools": "human_approval_required"}
runtime_contract


<div style="margin: 18px 0 12px; padding: 13px 15px; border-left: 4px solid #1976d2; background: #eef6ff; border-radius: 6px; color: #202124; box-shadow: 0 1px 2px rgba(60,64,67,.12);">
  <div style="font-size: .82em; font-weight: 700; color: #1976d2; letter-spacing: .02em;">📘  개념 · 실습 단계</div>
  <div style="margin-top: 8px; font-size: .94em; line-height: 1.6;">
<h2 style="margin: 0 0 8px; font-size: 1.05em; line-height: 1.35; color: #202124;">5. 외부 runtime 설정</h2>
<p style="margin: 7px 0;">실제 연결에는 LiteLLM endpoint, API key, 모델명, MCP Registry 권한이 필요합니다. 값은 출력하지 않고 설정 여부만 확인합니다.</p>
  </div>
</div>


In [ ]:
connection_contract = {"HARNESS_LLM_BASE_URL": "LiteLLM endpoint", "HARNESS_LLM_API_KEY": "secret", "HARNESS_MODEL": "deployed model", "MCP_REGISTRY": "approved tool registry"}
connection_contract


다음 구현 단계는 외부 Harness 소스의 `backend/integrations/deep_agent.py` 설정 계약과 맞춰 adapter를 만드는 것입니다. 이 저장소 밖의 MCP Registry, LiteLLM, n8n, HANA를 자동으로 호출하거나 생성하지 않습니다. 연결 대상과 권한이 확정된 후 진행하세요.